Linear Regression with Lasso, Ridge, and Elastic Net Regression

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import Lasso, LassoCV
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

In [79]:
#Import smallest dataset to be appended to other datasets
#This dataset is measure of a few simple totals that work as a representation of the logistical complexity of each airport
domestic_data_2024 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\T100_Domestic_Market_and_Segment_Data_8942359590531559889.csv")
domestic_data_2024.drop(["year", "enplanements", "arrivals", "OBJECTID"], axis=1, inplace=True) #Redundant with other columns
domestic_data_2024.head()

,origin,passengers,departures,freight,mail
0,01A,17,5,0,0
1,05A,1,1,0,0
2,06A,55,67,139,0
3,09A,43,15,0,0
4,1B1,32,7,0,0


In [80]:
#Import first dataset, 2015 Flight Delay Data
#Columns 7,8 needed dtype specified directly as infer failed

flights_2015 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2015_Delay_Data\flights.csv", dtype={"DESTINATION_AIRPORT":str, "ORIGIN_AIRPORT":str})

flights_2015.fillna({"AIR_SYSTEM_DELAY":0,"SECURITY_DELAY":0,"AIRLINE_DELAY":0,"LATE_AIRCRAFT_DELAY":0,"WEATHER_DELAY":0}, inplace=True)
flights_2015.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,NaN,0.0,0.0,0.0,0.0,0.0


In [81]:
#Append airport logistics numbers to the 2015 Flight Delay Dataset
domestic_data_2024.rename({"origin" : "ORIGIN_AIRPORT"}, inplace=True, axis=1)
flight_2015_extended = pd.merge(flights_2015, domestic_data_2024, how='left', on="ORIGIN_AIRPORT")
flight_2015_extended.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,passengers,departures,freight,mail
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,NaN,0.0,0.0,0.0,0.0,0.0,2702278.0,71765.0,3.116064e+09,95095127.0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,NaN,0.0,0.0,0.0,0.0,0.0,17666714.0,138571.0,2.182348e+08,8795256.0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,NaN,0.0,0.0,0.0,0.0,0.0,26340206.0,206637.0,8.440161e+08,47992961.0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,NaN,0.0,0.0,0.0,0.0,0.0,22288303.0,191107.0,3.490318e+08,14452644.0


In [82]:
#To perform any linear regression we need to drop non-numeric columns, columns that breakdown the delay into reasons, and unhelpful columns such as Year
#The non-numeric columns need dropped due to the number of unique values and the lack of meaningful directionality of those values
#Also need to drop Elapsed Time which is Air Time plus Taxiing time at both airports as they directly calculate the delay

flight_2015_delay_vars = flight_2015_extended.drop(["YEAR", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DIVERTED", "ELAPSED_TIME", "AIR_TIME", "TAXI_IN", "TAXI_OUT",
                                                          "CANCELLED", "CANCELLATION_REASON", "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", 
                                                          "WEATHER_DELAY"], axis=1)
flight_2015_delay_vars.dropna(axis=0, inplace=True)

target = flight_2015_delay_vars["ARRIVAL_DELAY"]
variables = flight_2015_delay_vars.drop("ARRIVAL_DELAY", axis=1)

In [83]:
#Scale features
scaler = StandardScaler()
variables_scl = scaler.fit_transform(variables)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl, target, random_state=42)

In [7]:
#Optimize Lasso Regression
alphas = np.logspace(-3,3,5)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_train, y_train)
print(f"Best performance alpha = {lasso_cv.alpha_:.5f}")

#Utilize Best Alpha
lasso_model = Lasso(alpha=lasso_cv.alpha_).fit(X_train,y_train)
train_preds = lasso_model.predict(X_train)
test_preds = lasso_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

Best performance alpha = 0.00100
The RMSE on the training data was 12.6234 minutes and on the test data was 12.6780 minutes


In [14]:
print("Variables\t\tWeights")
print("-"*30)

for v,c in zip(variables.columns, lasso_model.coef_):
    print(f"{v:<20}\t{c:.6f}")

Variables		Weights
------------------------------
MONTH               	-0.579913
DAY                 	-0.120543
DAY_OF_WEEK         	-0.262438
SCHEDULED_DEPARTURE 	-0.934809
DEPARTURE_TIME      	-2.573210
DEPARTURE_DELAY     	37.711967
WHEELS_OFF          	3.301809
SCHEDULED_TIME      	-11.840617
DISTANCE            	9.888863
WHEELS_ON           	0.627146
SCHEDULED_ARRIVAL   	-0.638708
ARRIVAL_TIME        	0.229026
passengers          	-2.215009
departures          	1.842948
freight             	-0.027282
mail                	-0.006665


In [16]:
#Optimize Ridge Regression
ridge_cv = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
print(f"Best performance alpha = {ridge_cv.alpha_:.5f}")

#Utilize Best Alpha
ridge_model = Ridge(alpha=ridge_cv.alpha_).fit(X_train,y_train)
train_preds = ridge_model.predict(X_train)
test_preds = ridge_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

Best performance alpha = 1.00000
The RMSE on the training data was 12.6234 minutes and on the test data was 12.6780 minutes


In [17]:
print("Variables\t\tWeights")
print("-"*30)

for v,c in zip(variables.columns, ridge_model.coef_):
    print(f"{v:<20}\t{c:.6f}")

Variables		Weights
------------------------------
MONTH               	-0.580930
DAY                 	-0.121539
DAY_OF_WEEK         	-0.263383
SCHEDULED_DEPARTURE 	-0.935025
DEPARTURE_TIME      	-2.610952
DEPARTURE_DELAY     	37.713691
WHEELS_OFF          	3.340363
SCHEDULED_TIME      	-11.925815
DISTANCE            	9.975240
WHEELS_ON           	0.629662
SCHEDULED_ARRIVAL   	-0.646283
ARRIVAL_TIME        	0.233091
passengers          	-2.286578
departures          	1.916358
freight             	-0.028728
mail                	-0.011997


Even trying to optimize the alphas, Lasso and Ridge regression both produce similar results to just standard Linear regression (RMSE of 12.64 minutes on all data). It is a positive that the test results closely match the training results.

In [60]:
#Optimize Elastic Net
elastic_net_model = ElasticNet()
params = {
    "alpha" : [.5,1,2],
    "l1_ratio" : [.1,.3,.5,.7,.9]
}

elastic_net_cv = GridSearchCV(elastic_net_model, param_grid = params).fit(X_train, y_train)
print(f"Best performance params = {elastic_net_cv.best_params_}")

Best performance params = {'alpha': 0.5, 'l1_ratio': 0.9}


In [84]:
elastic_net_model = ElasticNet(alpha = .5, l1_ratio=.9)
elastic_net_model.fit(X_train,y_train)

train_preds = elastic_net_model.predict(X_train)
test_preds = elastic_net_model.predict(X_test)

train_rmse = metrics.root_mean_squared_error(y_train, train_preds)
test_rmse = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

The RMSE on the training data was 12.9883 minutes and on the test data was 13.0374 minutes


In [85]:
print("Variables\t\tWeights")
print("-"*30)

for v,c in zip(variables.columns, elastic_net_model.coef_):
    print(f"{v:<20}\t{c:.6f}")

Variables		Weights
------------------------------
MONTH               	-0.159066
DAY                 	-0.000000
DAY_OF_WEEK         	-0.000000
SCHEDULED_DEPARTURE 	0.000000
DEPARTURE_TIME      	0.000000
DEPARTURE_DELAY     	35.431511
WHEELS_OFF          	0.000000
SCHEDULED_TIME      	-1.584244
DISTANCE            	-0.000000
WHEELS_ON           	0.000000
SCHEDULED_ARRIVAL   	0.000000
ARRIVAL_TIME        	0.000000
passengers          	-0.000000
departures          	-0.000000
freight             	0.000000
mail                	0.000000


Simply trying to use a linear regression model to predict flight delays like this is not feasible and not the ultimate goal of the models I will be building for this project. Looking at the coefficient weights this model fits a similar pattern to standard linear regression. It takes the departure delay time as the bulk of its arrival delay time estimation. This is a reasonable estimate for most flights because most flights are approximately on time. The RMSE of 12.99 minutes is very similar to the 12.53 minutes RSME of standard linear regression.

In [64]:
#Import 2nd Dataset, 2019 Flight Delay Info

delays_2019 = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\2019_Delay_Data\full_data_flightdelay.csv")

delays_2019.head()

,MONTH,DAY_OF_WEEK,DEP_DEL15,DEP_TIME_BLK,DISTANCE_GROUP,SEGMENT_NUMBER,CONCURRENT_FLIGHTS,NUMBER_OF_SEATS,CARRIER_NAME,AIRPORT_FLIGHTS_MONTH,...,PLANE_AGE,DEPARTING_AIRPORT,LATITUDE,LONGITUDE,PREVIOUS_AIRPORT,PRCP,SNOW,SNWD,TMAX,AWND
0,1,7,0,0800-0859,2,1,25,143,Southwest Airlines Co.,13056,...,8,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
1,1,7,0,0700-0759,7,1,29,191,Delta Air Lines Inc.,13056,...,3,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
2,1,7,0,0600-0659,7,1,27,199,Delta Air Lines Inc.,13056,...,18,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
3,1,7,0,0600-0659,9,1,27,180,Delta Air Lines Inc.,13056,...,2,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91
4,1,7,0,0001-0559,7,1,10,182,Spirit Air Lines,13056,...,1,McCarran International,36.08,-115.152,NONE,0.0,0.0,0.0,65.0,2.91


In [65]:
#The target variable for this dataset is "DEP_DEL15" it is a binary 1 if the flight was delayed more than 15 minutes in its departure delay else 0
#Predicting this with a linear model is inappropriate, but I will do so anyways for the excercise
num_delays_2019 = delays_2019.drop(["DEP_TIME_BLK", "CARRIER_NAME", "DEPARTING_AIRPORT", "PREVIOUS_AIRPORT"], axis = 1)

target_2019 = num_delays_2019["DEP_DEL15"]
variables_2019 = num_delays_2019.drop(["DEP_DEL15"], axis = 1)

In [66]:
#Scale features
scaler = StandardScaler()
variables_scl_2019 = scaler.fit_transform(variables_2019)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl_2019, target_2019, random_state=42)

In [22]:
#Optimize Lasso Regression
alphas = np.logspace(-3,3,5)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_train, y_train)
print(f"Best performance alpha = {lasso_cv.alpha_:.5f}")

#Utilize Best Alpha
lasso_model = Lasso(alpha=lasso_cv.alpha_).fit(X_train,y_train)
train_preds = lasso_model.predict(X_train)
test_preds = lasso_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} and on the test data was {test_rmse:.4f}")

Best performance alpha = 0.00100
The RMSE on the training data was 0.3857 and on the test data was 0.3859


In [21]:
print("Variables\t\t\t\tWeights")
print("-"*50)

for v,c in zip(variables_2019.columns, lasso_model.coef_):
    print(f"{v:<25}\t\t{c:.6f}")

Variables				Weights
--------------------------------------------------
MONTH                    		-0.004690
DAY_OF_WEEK              		0.000000
DISTANCE_GROUP           		0.015655
SEGMENT_NUMBER           		0.052144
CONCURRENT_FLIGHTS       		-0.004814
NUMBER_OF_SEATS          		0.013160
AIRPORT_FLIGHTS_MONTH    		0.012339
AIRLINE_FLIGHTS_MONTH    		0.000000
AIRLINE_AIRPORT_FLIGHTS_MONTH		-0.002993
AVG_MONTHLY_PASS_AIRPORT 		0.000000
AVG_MONTHLY_PASS_AIRLINE 		-0.002367
FLT_ATTENDANTS_PER_PASS  		-0.000000
GROUND_SERV_PER_PASS     		-0.004133
PLANE_AGE                		0.001872
LATITUDE                 		-0.000088
LONGITUDE                		0.013203
PRCP                     		0.027986
SNOW                     		0.014583
SNWD                     		0.006378
TMAX                     		0.002679
AWND                     		0.014612


In [22]:
#Optimize Ridge Regression
ridge_cv = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
print(f"Best performance alpha = {ridge_cv.alpha_:.5f}")

#Utilize Best Alpha
ridge_model = Ridge(alpha=ridge_cv.alpha_).fit(X_train,y_train)
train_preds = ridge_model.predict(X_train)
test_preds = ridge_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} and on the test data was {test_rmse:.4f}")

Best performance alpha = 31.62278
The RMSE on the training data was 0.3856 and on the test data was 0.3858


In [24]:
print("Variables\t\t\t\tWeights")
print("-"*50)

for v,c in zip(variables_2019.columns, ridge_model.coef_):
    print(f"{v:<25}\t\t{c:.6f}")

Variables				Weights
--------------------------------------------------
MONTH                    		-0.006902
DAY_OF_WEEK              		0.000526
DISTANCE_GROUP           		0.016974
SEGMENT_NUMBER           		0.053232
CONCURRENT_FLIGHTS       		-0.010986
NUMBER_OF_SEATS          		0.021941
AIRPORT_FLIGHTS_MONTH    		0.030927
AIRLINE_FLIGHTS_MONTH    		0.020951
AIRLINE_AIRPORT_FLIGHTS_MONTH		-0.005227
AVG_MONTHLY_PASS_AIRPORT 		-0.011016
AVG_MONTHLY_PASS_AIRLINE 		-0.027704
FLT_ATTENDANTS_PER_PASS  		-0.000040
GROUND_SERV_PER_PASS     		-0.001990
PLANE_AGE                		0.005477
LATITUDE                 		-0.002179
LONGITUDE                		0.015914
PRCP                     		0.028749
SNOW                     		0.015344
SNWD                     		0.007665
TMAX                     		0.002915
AWND                     		0.015214


Once again the model is inappropriate for predicting a binary outcome. Also again the results are in-line with the results of normal linear regression (RMSE of .3856 on all data)

In [67]:
#Optimize Elastic Net
elastic_net_model = ElasticNet()
params = {
    "alpha" : [.5,1,2],
    "l1_ratio" : [.1,.3,.5,.7,.9]
}

elastic_net_cv = GridSearchCV(elastic_net_model, param_grid = params).fit(X_train, y_train)
print(f"Best performance params = {elastic_net_cv.best_params_}")

Best performance params = {'alpha': 0.5, 'l1_ratio': 0.1}


In [68]:
elastic_net_model = ElasticNet(alpha = .5, l1_ratio = .1)
elastic_net_model.fit(X_train,y_train)

train_preds = elastic_net_model.predict(X_train)
test_preds = elastic_net_model.predict(X_test)

train_rmse = metrics.root_mean_squared_error(y_train, train_preds)
test_rmse = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

The RMSE on the training data was 0.3916 minutes and on the test data was 0.3918 minutes


In [69]:
print("Variables\t\t\tWeights")
print("-"*40)

for v,c in zip(variables_2019.columns, elastic_net_model.coef_):
    print(f"{v:<25}\t{c:.6f}")

Variables			Weights
----------------------------------------
MONTH                    	0.000000
DAY_OF_WEEK              	0.000000
DISTANCE_GROUP           	0.000000
SEGMENT_NUMBER           	0.000000
CONCURRENT_FLIGHTS       	0.000000
NUMBER_OF_SEATS          	0.000000
AIRPORT_FLIGHTS_MONTH    	0.000000
AIRLINE_FLIGHTS_MONTH    	0.000000
AIRLINE_AIRPORT_FLIGHTS_MONTH	0.000000
AVG_MONTHLY_PASS_AIRPORT 	0.000000
AVG_MONTHLY_PASS_AIRLINE 	0.000000
FLT_ATTENDANTS_PER_PASS  	0.000000
GROUND_SERV_PER_PASS     	0.000000
PLANE_AGE                	0.000000
LATITUDE                 	0.000000
LONGITUDE                	0.000000
PRCP                     	0.000000
SNOW                     	0.000000
SNWD                     	0.000000
TMAX                     	0.000000
AWND                     	0.000000


Not alot to say about these results. An RMSE of .3914 is very similar to the standard linear regression result of an RMSE of .3856. Neither is a good error for predicting a binary, but neither is an appropriate model either.

In [70]:
#Import 3rd Dataset

delay_causes = pd.read_csv(r"C:\Users\lemrd\Downloads\OMDS\Mod B\Sem 2\Data\Flight_Delay_and_Causes_Data\Flight_delay.csv")
#Remove the few duplicate datapoints
delay_causes.drop_duplicates(keep="first", inplace=True)
#Drop columns with no information (all values are the same)
delay_causes.drop(["Cancelled","Diverted","CancellationCode"],axis=1, inplace=True)
#Fix long delays to break out of 24 hour time to show the real delay length
data_to_shift = delay_causes[delay_causes["ArrTime"] < (delay_causes["CRSArrTime"] - 100)].index
delay_causes.loc[data_to_shift,"ArrTime"] = delay_causes.loc[data_to_shift,"ArrTime"] + 2400
                           
delay_causes.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,Dest,Dest_Airport,Distance,TaxiIn,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,BWI,Baltimore-Washington International Airport,515,3,10,2,0,0,0,32
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,LAS,McCarran International Airport,1591,3,7,10,0,0,0,47
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,MCO,Orlando International Airport,828,6,8,8,0,0,0,72
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,PHX,Phoenix Sky Harbor International Airport,1489,7,8,3,0,0,0,12
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,TPA,Tampa International Airport,838,4,9,0,0,0,0,16


In [71]:
#Append Logistics Dataset
domestic_data_2024.rename({"ORIGIN_AIRPORT" : "Origin"}, inplace=True, axis=1)
delay_causes_extended = pd.merge(delay_causes, domestic_data_2024, how='left', on="Origin")
delay_causes_extended.head()

,DayOfWeek,Date,DepTime,ArrTime,CRSArrTime,UniqueCarrier,Airline,FlightNum,TailNum,ActualElapsedTime,...,TaxiOut,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay,passengers,departures,freight,mail
0,4,03-01-2019,1829,1959,1925,WN,Southwest Airlines Co.,3920,N464WN,90,...,10,2,0,0,0,32,5194000.0,60772.0,842478043.0,84741.0
1,4,03-01-2019,1937,2037,1940,WN,Southwest Airlines Co.,509,N763SW,240,...,7,10,0,0,0,47,5194000.0,60772.0,842478043.0,84741.0
2,4,03-01-2019,1644,1845,1725,WN,Southwest Airlines Co.,1333,N334SW,121,...,8,8,0,0,0,72,5194000.0,60772.0,842478043.0,84741.0
3,4,03-01-2019,1452,1640,1625,WN,Southwest Airlines Co.,675,N286WN,228,...,8,3,0,0,0,12,5194000.0,60772.0,842478043.0,84741.0
4,4,03-01-2019,1323,1526,1510,WN,Southwest Airlines Co.,4,N674AA,123,...,9,0,0,0,0,16,5194000.0,60772.0,842478043.0,84741.0


In [72]:
#Drop non-numeric columns
num_delay_causes = delay_causes_extended.drop(["Date", "UniqueCarrier", "Airline", "TailNum", "Origin", "Org_Airport", "Dest", "Dest_Airport"], axis = 1)
#Drop missing data
num_delay_causes.dropna(axis=0, inplace=True)

target_delay = num_delay_causes["ArrDelay"]
variables_delay = num_delay_causes.drop(["ArrDelay","CarrierDelay", "WeatherDelay",  "NASDelay", "SecurityDelay", "LateAircraftDelay",
                                         "AirTime", "ArrTime","ActualElapsedTime"], axis = 1)

In [73]:
#Scale features
scaler = StandardScaler()
variables_scl_delay = scaler.fit_transform(variables_delay)

#Train test split
X_train, X_test, y_train, y_test = train_test_split(variables_scl_delay, target_delay, random_state=42)

In [30]:
#Optimize Lasso Regression
alphas = np.logspace(-3,3,5)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_train, y_train)
print(f"Best performance alpha = {lasso_cv.alpha_:.5f}")

#Utilize Best Alpha
lasso_model = Lasso(alpha=lasso_cv.alpha_).fit(X_train,y_train)
train_preds = lasso_model.predict(X_train)
test_preds = lasso_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

Best performance alpha = 0.00100
The RMSE on the training data was 10.7216 minutes and on the test data was 10.6951 minutes


In [33]:
print("Variables\t\t\tWeights")
print("-"*40)

for v,c in zip(variables_2019.columns, lasso_model.coef_):
    print(f"{v:<25}\t{c:.6f}")

Variables			Weights
----------------------------------------
MONTH                    	-0.121981
DAY_OF_WEEK              	-0.203925
DISTANCE_GROUP           	-0.315781
SEGMENT_NUMBER           	-0.017914
CONCURRENT_FLIGHTS       	-14.687101
NUMBER_OF_SEATS          	54.307010
AIRPORT_FLIGHTS_MONTH    	13.030331
AIRLINE_FLIGHTS_MONTH    	4.832708
AIRLINE_AIRPORT_FLIGHTS_MONTH	14.072743
AVG_MONTHLY_PASS_AIRPORT 	-2.653325
AVG_MONTHLY_PASS_AIRLINE 	0.995372
FLT_ATTENDANTS_PER_PASS  	-0.475952
GROUND_SERV_PER_PASS     	0.307725


In [34]:
#Optimize Ridge Regression
ridge_cv = RidgeCV(alphas=alphas, cv=5).fit(X_train, y_train)
print(f"Best performance alpha = {ridge_cv.alpha_:.5f}")

#Utilize Best Alpha
ridge_model = Ridge(alpha=ridge_cv.alpha_).fit(X_train,y_train)
train_preds = ridge_model.predict(X_train)
test_preds = ridge_model.predict(X_test)
train_rmse = metrics.root_mean_squared_error(y_train,train_preds)
test_rmse = metrics.root_mean_squared_error(y_test,test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} and on the test data was {test_rmse:.4f}")

Best performance alpha = 0.00100
The RMSE on the training data was 10.7215 and on the test data was 10.6950


In [36]:
print("Variables\t\t\tWeights")
print("-"*40)

for v,c in zip(variables_2019.columns, ridge_model.coef_):
    print(f"{v:<25}\t{c:.6f}")

Variables			Weights
----------------------------------------
MONTH                    	-0.122716
DAY_OF_WEEK              	-0.204074
DISTANCE_GROUP           	-0.316246
SEGMENT_NUMBER           	-0.018427
CONCURRENT_FLIGHTS       	-14.764106
NUMBER_OF_SEATS          	54.308422
AIRPORT_FLIGHTS_MONTH    	13.106942
AIRLINE_FLIGHTS_MONTH    	4.835183
AIRLINE_AIRPORT_FLIGHTS_MONTH	14.076179
AVG_MONTHLY_PASS_AIRPORT 	-2.710697
AVG_MONTHLY_PASS_AIRLINE 	1.053257
FLT_ATTENDANTS_PER_PASS  	-0.479516
GROUND_SERV_PER_PASS     	0.307138


Standard linear regression on this dataset produced an RMSE of 10.71 minutes. There is not improvement shown on this dataset either. This dataset is very similar to the first so this outcome is not at all surprising to me.

In [74]:
#Optimize Elastic Net
elastic_net_model = ElasticNet()
params = {
    "alpha" : [.5,1,2],
    "l1_ratio" : [.1,.3,.5,.7,.9]
}

elastic_net_cv = GridSearchCV(elastic_net_model, param_grid = params).fit(X_train, y_train)
print(f"Best performance params = {elastic_net_cv.best_params_}")

Best performance params = {'alpha': 0.5, 'l1_ratio': 0.9}


In [75]:
elastic_net_model = ElasticNet(alpha = .5, l1_ratio = .9)
elastic_net_model.fit(X_train,y_train)

train_preds = elastic_net_model.predict(X_train)
test_preds = elastic_net_model.predict(X_test)

train_rmse = metrics.root_mean_squared_error(y_train, train_preds)
test_rmse = metrics.root_mean_squared_error(y_test, test_preds)

print(f"The RMSE on the training data was {train_rmse:.4f} minutes and on the test data was {test_rmse:.4f} minutes")

The RMSE on the training data was 11.4883 minutes and on the test data was 11.4462 minutes


In [76]:
print("Variables\t\tWeights")
print("-"*35)

for v,c in zip(variables_delay.columns, elastic_net_model.coef_):
    print(f"{v:<20}\t{c:.6f}")

Variables		Weights
-----------------------------------
DayOfWeek           	-0.000000
DepTime             	-0.000000
CRSArrTime          	-0.055516
FlightNum           	0.000000
CRSElapsedTime      	-0.997900
DepDelay            	51.147413
Distance            	-0.000000
TaxiIn              	4.032259
TaxiOut             	12.298163
passengers          	-1.111404
departures          	-0.052616
freight             	-0.000000
mail                	0.000000


RMSE for standard linear regression was 10.71 minutes, while elastic net had RMSE of 10.72 minutes. I am surprised that this model did not drop the Flight Number. The results overall are still much the same as linear regression where the same few variables dominated the model. In this case, it is the expected elapsed flight time plus the taxiing times.